# 🔁 Ejercicios: Stripe
# Suscripciones SaaS - MRR, antigüedad y churn

⚠️ Datos SIMULADOS / ilustrativos
(generados con pandas + numpy).
No son cifras oficiales de Stripe.

### 💼 PREGUNTA ENTREVISTA:
### ¿Cuál es el churn
### global y cuánto MRR
### nos cuesta?

Churn = % de clientes cancelados.
La métrica reina del SaaS.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/'
  'stripe_suscripciones.csv'
)

canc = df['estado'] == 'Cancelado'
churn = canc.mean() * 100

mrr_perd = df.loc[
  canc, 'mrr_usd'
].sum()

print(f'Churn global: '
      f'{churn:.1f}%')
print(f'MRR perdido: '
      f'{mrr_perd:,.0f} USD/mes')

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué plan retiene
### mejor a sus
### clientes?

Churn rate por plan. ¿Los planes
caros fidelizan más?

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/'
  'stripe_suscripciones.csv'
)

df['canc'] = (
  df['estado'] == 'Cancelado'
).astype(int)

r = (
  df
  .groupby('plan')
  .agg(
    churn_pct=('canc',
      lambda s: s.mean() * 100),
    mrr=('mrr_usd', 'mean'),
    n=('cliente_id', 'count'),
  )
  .round(1)
  .sort_values('churn_pct')
)

print('Churn por plan:')
print(r)

### 💼 PREGUNTA ENTREVISTA:
### ¿La antigüedad protege
### del churn? (lift por
### tramos)

Troceamos antigüedad y vemos cómo
cae el churn frente a la media.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/'
  'stripe_suscripciones.csv'
)

df['canc'] = (
  df['estado'] == 'Cancelado'
).astype(int)

df['tramo'] = pd.cut(
  df['antiguedad_meses'],
  [0, 6, 12, 24, 48],
  labels=['0-6m', '6-12m',
    '1-2a', '2a+'],
)

base = df['canc'].mean()
r = (
  df
  .groupby('tramo', observed=True)
  ['canc']
  .mean()
)

lift = (r / base).round(2)
print('Lift de churn por antigüedad:')
print(lift)

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué canal trae los
### clientes que más
### aguantan? (LTV)

LTV = MRR x antigüedad. Mide la
calidad de cada canal de captación.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/'
  'stripe_suscripciones.csv'
)

df['canc'] = (
  df['estado'] == 'Cancelado'
).astype(int)

r = (
  df
  .groupby('canal')
  .agg(
    ltv=('ltv_usd', 'mean'),
    churn_pct=('canc',
      lambda s: s.mean() * 100),
  )
  .round(1)
  .sort_values('ltv',
    ascending=False)
)

print('LTV y churn por canal:')
print(r)

### 💼 PREGUNTA ENTREVISTA:
### ¿Por qué se van?
### (motivos de
### cancelación)

Reparto de motivos entre los
cancelados. Dónde atacar primero.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/'
  'stripe_suscripciones.csv'
)

canc = df[
  df['estado'] == 'Cancelado'
]

g = (
  canc['motivo_cancelacion']
  .value_counts()
)

pct = (g / g.sum() * 100).round(1)

tabla = pd.DataFrame({
  'n': g,
  '%': pct,
})

print('Motivos de cancelación:')
print(tabla)